In [ ]:
import pandas as pd
from collections import defaultdict

In [ ]:
data_dir = "./data/"
raw_data_dir = "./data/raw/"
transformed_data_dir = "./data/transformed/"

# Store raw data as CSV

In [ ]:
file = f'{raw_data_dir}Digitalhaushalt_Open_Data_DOI.xlsx'

types = defaultdict(lambda: str, A="int", B="float")
df_raw = pd.read_excel(file, sheet_name='Daten', dtype=types)

df_raw.info()

df_raw.to_csv(f'{raw_data_dir}digitalhaushalt_raw.csv', index=False, encoding='utf-8')

# Clean data

In [ ]:
file_csv = f'{raw_data_dir}digitalhaushalt_raw.csv'
types = defaultdict(lambda: str, A="int", B="float")
df_csv = pd.read_csv(file_csv, dtype=types)

df_csv.info()

In [ ]:
# change datatype of some float to integer
list_of_columns_integer = [
                   '_1_infra',
                   '_2_dig_wirtschaft',
                   '_3_dig_verwaltung',
                   '_4_dig_kompetenzen',
                   '_5_dig_kultur',
                   '_6_forschung_inno',
                   '_7_gesundheitswesen',
                   '_8_bundeswehr',
                   '_9_unteilbare_ausg',
                   'any_tag',
                   'expert_tag',
                   'large_soll_tag',
                   'ml_tag',
                   'digital_series_tag',
                   'texan_tag'
                   ]
df_csv[list_of_columns_integer] = df_csv[list_of_columns_integer].astype(float).astype('Int64')

list_of_columns_float = ['soll', 'digi_soll_eng','digi_soll_weit','ist','digi_ist_eng','digi_ist_weit']

df_csv[list_of_columns_float] = df_csv[list_of_columns_float].astype(float).astype('float')

df_csv.info()

In [ ]:
import numpy as np
import duckdb

funktionen = duckdb.query(
    'select distinct' \
        '"funktion"' \
    'from df_csv ' \
    'where cast(funktion as string) like \'%0\' ' \
    'order by 1').df()


print(funktionen)

mask = (
    df_csv['funktion'].notna()
    & df_csv['funktion'].astype('Int64').ge(100)
    & (df_csv['funktion'].astype('Int64') % 10 == 0)
)

df_csv['funktion'] = np.where(
    mask,
    (df_csv['funktion'].astype('Int64') // 10).astype('str'),
    df_csv['funktion'].astype('str')
)

funktionen = duckdb.query(
    'select distinct' \
        '"funktion"' \
    'from df_csv ' \
    'where cast(funktion as string) like \'%0\' ' \
    'order by 1').df()

print(funktionen)

In [ ]:
gruppen = duckdb.query(
    'select distinct' \
        '"gruppe"' \
    'from df_csv ' \
    'where cast(gruppe as string) like \'%0\' ' \
    'order by 1').df()


print(gruppen)

mask = (
    df_csv['gruppe'].notna()
    & df_csv['gruppe'].astype('Int64').ge(100)
    & (df_csv['gruppe'].astype('Int64') % 10 == 0)
)

df_csv['gruppe'] = np.where(
    mask,
    (df_csv['gruppe'].astype('Int64') // 10).astype('str'),
    df_csv['gruppe'].astype('str')
)

gruppen = duckdb.query(
    'select distinct' \
        '"gruppe"' \
    'from df_csv ' \
    'where cast(gruppe as string) like \'%0\' ' \
    'order by 1').df()

print(gruppen)

# Add mapping

In [ ]:
df_einzelplan_mapping = pd.read_csv(f'{raw_data_dir}einzelplan_mapping.csv', dtype='str')
df_kapitel_mapping = pd.read_csv(f'{raw_data_dir}kapitel_mapping.csv', dtype='str')
df_gruppierung_mapping = pd.read_csv(f'{raw_data_dir}gruppierung_mapping.csv', dtype='str')
df_funktionen_mapping = pd.read_csv(f'{raw_data_dir}funktionen_mapping.csv', dtype='str')
df_digi_klasse_mapping = pd.read_csv(f'{raw_data_dir}digi_klasse_mapping.csv', dtype='str')

df_csv = df_csv.merge(
    df_einzelplan_mapping[['einzelplan', 'einzelplan-text']],
    on='einzelplan',
    how='left'
)

print(f'{df_csv.shape} after einzelplan mapping')

df_csv = df_csv.merge(
    df_kapitel_mapping[['kapitel', 'kapitel-text']],
    on='kapitel',
    how='left'
)

print(f'{df_csv.shape} after kapitel mapping')

# df_csv["gruppe"] = df_csv["gruppe"].astype("Int64")
# Gruppe
df_gruppierung_mapping["code"] = df_gruppierung_mapping["code"].str.split("/").str[0]
df_csv = df_csv.merge(
    df_gruppierung_mapping[["code", "bezeichnung"]],
    left_on="gruppe",
    right_on="code",
    how="left"
).rename(columns={"bezeichnung": "gruppe-text"}).drop(columns=["code"])
# Kategorien kürzen
df_csv["gruppe-text"] = df_csv["gruppe-text"].apply(lambda x: x[:100] if isinstance(x, str) else x)

print(f'{df_csv.shape} after gruppe mapping')

# Hauptgruppe
df_hauptgruppe = df_gruppierung_mapping[df_gruppierung_mapping["ebene"] == "Hauptgruppe"]
df_csv["hauptgruppe"] = df_csv["gruppe"].str[0]
print(df_csv["hauptgruppe"])
print(df_hauptgruppe)
df_csv = df_csv.merge(
    df_hauptgruppe[["code", "bezeichnung"]],
    left_on="hauptgruppe",
    right_on="code",
    how="left"
).rename(columns={"bezeichnung": "hauptgruppe-text"}).drop(columns=["code", "hauptgruppe"])
# Kategorien kürzen
df_csv["hauptgruppe-text"] = df_csv["hauptgruppe-text"].apply(lambda x: x[:100] if isinstance(x, str) else x)

print(f'{df_csv.shape} after hauptgruppe mapping')

# Obergruppe
df_obergruppe = df_gruppierung_mapping[df_gruppierung_mapping["ebene"] == "Obergruppe"]
df_csv["obergruppe"] = df_csv["gruppe"].str[0:2]
print(df_csv["obergruppe"])
df_csv = df_csv.merge(
    df_obergruppe[["code", "bezeichnung"]],
    left_on="obergruppe",
    right_on="code",
    how="left"
).rename(columns={"bezeichnung": "obergruppe-text"}).drop(columns=["code", "obergruppe"])
# Kategorien kürzen
df_csv["obergruppe-text"] = df_csv["obergruppe-text"].apply(lambda x: x[:100] if isinstance(x, str) else x)

print(f'{df_csv.shape} after obergruppe mapping')

df_funktionen_mapping["code"] = df_funktionen_mapping["code"]
df_csv = df_csv.merge(
    df_funktionen_mapping[['code', 'bezeichnung']],
    left_on='funktion',
    right_on='code',
    how='left'
).rename(columns={'bezeichnung': 'funktion-text'}).drop(columns=["code"])
# Kategorien kürzen
df_csv["funktion-text"] = df_csv["funktion-text"].apply(lambda x: x[:100] if isinstance(x, str) else x)

print(f'{df_csv.shape} after funktion mapping')

# Hauptfunktion
df_hauptfunktion = df_funktionen_mapping[df_funktionen_mapping["ebene"] == "Hauptfunktion"]
df_csv["hauptfunktion"] = df_csv["funktion"].str[0]
print(df_csv["hauptfunktion"])
print(df_hauptfunktion)
df_csv = df_csv.merge(
    df_hauptfunktion[["code", "bezeichnung"]],
    left_on="hauptfunktion",
    right_on="code",
    how="left"
).rename(columns={"bezeichnung": "hauptfunktion-text"}).drop(columns=["code", "hauptfunktion"])
# Kategorien kürzen
df_csv["hauptfunktion-text"] = df_csv["hauptfunktion-text"].apply(lambda x: x[:100] if isinstance(x, str) else x)

print(f'{df_csv.shape} after hauptfunktion mapping')

# Oberfunktion
df_oberfunktion = df_funktionen_mapping[df_funktionen_mapping["ebene"] == "Oberfunktion"]
df_csv["oberfunktion"] = df_csv["funktion"].str[0:2]
print(df_csv["oberfunktion"])
df_csv = df_csv.merge(
    df_oberfunktion[["code", "bezeichnung"]],
    left_on="oberfunktion",
    right_on="code",
    how="left"
).rename(columns={"bezeichnung": "oberfunktion-text"}).drop(columns=["code", "oberfunktion"])
# Kategorien kürzen
df_csv["oberfunktion-text"] = df_csv["oberfunktion-text"].apply(lambda x: x[:100] if isinstance(x, str) else x)

print(f'{df_csv.shape} after oberfunktion mapping')

df_digi_klasse_mapping["digi_klasse"] = df_digi_klasse_mapping["digi_klasse"]
df_csv = df_csv.merge(
    df_digi_klasse_mapping[['digi_klasse', 'digi_klasse-text']],
    on='digi_klasse',
    how='left'
)

print(f'{df_csv.shape} after digi_klasse mapping')

# boolean case when 1 then Kategorie Infrastruktur etc pp
df_csv["kategorie"] = np.select(
    [
        df_csv['_1_infra'].fillna(0) == 1,
        df_csv['_2_dig_wirtschaft'].fillna(0) == 1,
        df_csv['_3_dig_verwaltung'].fillna(0) == 1,
        df_csv['_4_dig_kompetenzen'].fillna(0) == 1,
        df_csv['_5_dig_kultur'].fillna(0) == 1,
        df_csv['_6_forschung_inno'].fillna(0) == 1,
        df_csv['_7_gesundheitswesen'].fillna(0) == 1,
        df_csv['_8_bundeswehr'].fillna(0) == 1,
        df_csv['_9_unteilbare_ausg'].fillna(0) == 1
    ],
    [
        'Infrastruktur',
        'Digitalisierung der Wirtschaft',
        'Digitalisierung der öffentlichen Verwaltung',
        'Digitalisierung im Bereich Kultur / Medien / Zivilgesellschaft',
        'Digitale Kultur',
        'Förderung von Forschung und Innovation',
        'Gesundheitswesen',
        'Bundeswehr',
        'Unteilbare Ausgaben'
    ],
    default=''
)


In [ ]:
df_csv.drop(columns=['id', 'titel_text'], inplace=True)

df_csv.to_csv(f'{transformed_data_dir}digitalhaushalt_transformed.csv', index=False, encoding='utf-8')

In [ ]:
df_csv.info()